In [ ]:
# ======================================================================
# CLEAN PPR DATA: one ML-ready table, one row per property sale
# ======================================================================
# Input:  processed_dataset\property_price_dataset\ppr_2021_onwards.csv
# Output: clean_dataset\ppr\ppr_clean.csv
#
# What this table is for (from the proposal, Student 3 + group model):
#   - Isolation Forest (unusual sales)   -> price, county, quarter, property type
#   - temporal analysis                  -> county x quarter median prices
#   - Polynomial regression              -> target: log_price
#   - Logistic regression                -> target: not_full_market_price
#   - group model                        -> log_price + county + quarter
#
# Rules for this notebook:
#   - fix the data (types, labels, Yes/No, county names)
#   - unusual prices are only FLAGGED, not deleted
#   - do NOT fill missing values, scale or encode here (done in the model Pipeline)
#
# Needs 00_county_lookup.ipynb to be run first (uses county_lookup.csv)
# ======================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet'

ppr_file = base + r'\processed_dataset\property_price_dataset\ppr_2021_onwards.csv'
clean    = base + r'\clean_dataset'

In [ ]:
ppr = pd.read_csv(ppr_file, low_memory=False)

print(ppr.shape)
ppr.head(3)

In [ ]:
# Each sale gets its own id, so it can be found again after joins and
# so flagged sales (e.g. from Isolation Forest) can be traced back.

ppr.insert(0, 'sale_id', range(1, len(ppr) + 1))

# Removing extra spaces from text, and turning blank text into missing
for col in ['address', 'county', 'eircode', 'not_full_market_price',
            'vat_exclusive', 'property_description']:
    ppr[col] = ppr[col].str.strip()
    ppr[col] = ppr[col].replace('', np.nan)

ppr['sale_date'] = pd.to_datetime(ppr['sale_date'])
print(ppr['sale_date'].min(), 'to', ppr['sale_date'].max())

In [ ]:
# County: using the shared county_lookup.csv, so the names and ids match
# BER, planning and rent exactly.

lookup = pd.read_csv(clean + r'\county_lookup.csv')
lookup = lookup[lookup['source'] == 'ppr'][['raw_name', 'county', 'county_id']]
lookup = lookup.rename(columns={'raw_name': 'county_raw'})

ppr = ppr.rename(columns={'county': 'county_raw'})
ppr = ppr.merge(lookup, on='county_raw', how='left')
ppr = ppr.drop(columns=['county_raw'])

print('Rows without a county:', ppr['county'].isnull().sum())   # should be 0
print(ppr['county'].nunique(), 'counties')

In [ ]:
# Property type: New or Second-Hand.
# A handful of rows are written in Irish:
#   'Teach/Árasán Cónaithe Nua'      = new home
#   'Teach/Árasán Cónaithe Atháimhe' = second-hand home

print(ppr['property_description'].value_counts())

ppr['property_type'] = 'Second-Hand'
ppr.loc[ppr['property_description'].str.contains('New'), 'property_type'] = 'New'
ppr.loc[ppr['property_description'].str.contains('Nua'), 'property_type'] = 'New'

print()
print(ppr['property_type'].value_counts())

In [ ]:
# Yes / No columns -> 1 / 0

ppr['not_full_market_price'] = ppr['not_full_market_price'].map({'Yes': 1, 'No': 0})
ppr['vat_exclusive'] = ppr['vat_exclusive'].map({'Yes': 1, 'No': 0})

print(ppr['not_full_market_price'].value_counts(dropna=False))
print(ppr['vat_exclusive'].value_counts(dropna=False))

In [ ]:
# Logistic regression target check: how rare are 'not full market price' sales?
# (imbalanced -> class weights / stratification at the modelling stage)

print(ppr['not_full_market_price'].value_counts(normalize=True).round(3))

In [ ]:
# Price: keeping the recorded price and the VAT-adjusted price (made in the
# processed notebook). The model target is the log of the VAT-adjusted price,
# because prices are very skewed and the proposal uses log price.

ppr = ppr.rename(columns={'price': 'price_recorded'})
ppr['log_price'] = np.log(ppr['price_incl_vat'])

print(ppr[['price_recorded', 'price_incl_vat', 'log_price']].describe().round(2))

In [ ]:
# Unusual prices: using the IQR rule on log price (same idea as the outlier
# notebook from class). These are only FLAGGED, not removed.
# A flagged price is not automatically wrong: some are real (expensive Dublin
# homes, cheap rural cottages), others are bulk sales (a whole block of
# apartments for one price) or family transfers. The proposal says unusual
# sales should be looked at, not deleted.

q1 = ppr['log_price'].quantile(0.25)
q3 = ppr['log_price'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

ppr['is_extreme_price'] = 0
ppr.loc[(ppr['log_price'] < lower) | (ppr['log_price'] > upper), 'is_extreme_price'] = 1

print('Lower limit: EUR', round(np.exp(lower)))
print('Upper limit: EUR', round(np.exp(upper)))
print('Flagged sales:', ppr['is_extreme_price'].sum(),
      '(', round(ppr['is_extreme_price'].mean() * 100, 1), '% )')

In [ ]:
# How the flag relates to 'not full market price' (many cheap sales are
# family transfers, which is what the logistic regression is about)

pd.crosstab(ppr['is_extreme_price'], ppr['not_full_market_price'], normalize='index').round(3)

In [ ]:
# Log price distribution with the flag limits

plt.figure(figsize=(10, 4))
ppr['log_price'].hist(bins=100)
plt.axvline(lower, color='red', linestyle='--')
plt.axvline(upper, color='red', linestyle='--')
plt.title('Log sale price (red lines = IQR limits)')
plt.xlabel('log(price incl. VAT)')
plt.show()

In [ ]:
# Eircode: the full code points at one exact home, so it is not kept.
# The first 3 characters (routing key, e.g. 'D03') are an area code, which
# could be a finer location than county if needed later. 28% are missing.

ppr['eircode_routing_key'] = ppr['eircode'].str[:3]

print('Missing routing key:', round(ppr['eircode_routing_key'].isnull().mean() * 100, 1), '%')
print(ppr['eircode_routing_key'].nunique(), 'routing keys')

In [ ]:
# Choosing the columns for the clean table.
# Left out on purpose:
#   eircode               -> identifies one home (routing key kept instead)
#   property_size         -> 100% empty since 2021
#   property_description  -> replaced by property_type
# The address is kept only to spot repeat sales of the same home; it is not
# a model input.

keep = ['sale_id', 'sale_date', 'year', 'quarter', 'county', 'county_id',
        'address', 'eircode_routing_key',
        'property_type', 'vat_exclusive',
        'price_recorded', 'price_incl_vat', 'log_price',
        'not_full_market_price', 'is_extreme_price']

ppr_clean = ppr[keep]

print(ppr_clean.shape)
ppr_clean.head()

In [ ]:
# Missing values left (% per column)

missing = (ppr_clean.isnull().mean() * 100).round(1)
missing[missing > 0]

In [ ]:
ppr_clean.info()

In [ ]:
# Saving the clean PPR table.

save_folder = clean + r'\ppr'
os.makedirs(save_folder, exist_ok=True)

ppr_clean.to_csv(save_folder + r'\ppr_clean.csv', index=False)
print('Saved', ppr_clean.shape, 'to', save_folder + r'\ppr_clean.csv')